# EFS - Feature Store

## Disclaimer
The sample code (“Sample Code”) provided is not covered by any Teradata agreements. Please be aware that Teradata has no control over the model responses to such sample code and such response may vary. The use of the model by Teradata is strictly for demonstration purposes and does not constitute any form of certification or endorsement. The sample code is provided “AS IS” and any express or implied warranties, including the implied warranties of merchantability and fitness for a particular purpose, are disclaimed. In no event shall Teradata be liable for any direct, indirect, incidental, special, exemplary, or consequential damages (including, but not limited to, procurement of substitute goods or services; loss of use, data, or profits; or business interruption) sustained by you or a third party, however caused and on any theory of liability, whether in contract, strict liability, or tort arising in any way out of the use of this sample code, even if advised of the possibility of such damage.

## Context 
**Enterprise Feature Store**

This notebook demonstrates how to build, manage, and utilize an enterprise feature store using TeradataML. It covers the end-to-end workflow for feature engineering, ingestion, cataloging, and governance of features derived from business data such as sales, marketing, and transactions.

**Notebook Purpose:**
- Show how to create and manage a centralized feature repository for analytics and machine learning.
- Demonstrate feature engineering and ingestion from raw business datasets.
- Illustrate feature lineage, versioning, and governance for reproducible ML workflows.

## 1. Import the required libraries

In [1]:
from teradataml import create_context, display
from getpass import getpass
from collections import OrderedDict
from teradatasqlalchemy import INTEGER, FLOAT, VARCHAR, DATE
from sqlalchemy import literal_column
display.enable_ui = False

## 2. Connect to Vantage with Admin user

Connecting to Vantage with an Admin user is required for initial setup tasks such as creating the feature store, configuring storage, and granting permissions to other users. These operations typically require elevated privileges.

In [2]:
ctx = create_context(config_file='admin_config_file.env')

## 3. Setup a Feature Store Repository

### 3.1. Create the FeatureStore

In [3]:
# Create a repo. 
fs = ctx.FeatureStore(repo='enterprise_feature_repo')

Repo enterprise_feature_repo does not exist. Run FeatureStore.setup() to create the repo and setup FeatureStore.


### 3.2. List the available feature stores. 

In [4]:
ctx.FeatureStore.list_repos()

repos
vfs_test
test_fs
テストリポジトリ
vfs_test_v1_obj


### 3.3. Setup the FeatureStore

In [5]:
fs.setup()

True

### 3.4. Grant the access to user

**Note:** 
Granting read/write access to a user is necessary so they can create, modify, and manage features and metadata within the feature store. This ensures the specified user has the required permissions to work with the feature store objects. If needed, you can later revoke these rights using `fs.revoke.read_write(username)`.

In [6]:
username = getpass(prompt = 'username: ')
fs.grant.read_write(username)

True

## 4. Connect to a Vantage system with user that has granted the permissions

### 4.1. Remove context with Admin user

In [7]:
# Disconnect from the system and connect to system again from the 'user'.
ctx.remove_context()

True

### 4.2. Create context with non-admin user

In [8]:
ctx = create_context(config_file='non_admin_config_file.env')

### 4.3. Create Feature Store object with non-admin user to ingest feature values

In [9]:
# First initiate feature store for the repo enterprise_feature_repo. Keep the data domain as analytics.
fs = ctx.FeatureStore(repo='enterprise_feature_repo', data_domain='Analytics')

FeatureStore is ready to use.


### Feature Engineering

<p>In this feature engineering process, we will use the <code>teradataml</code> package to create Teradata DataFrames that implement the required computations. The two main feature engineering tasks are:</p>

<ol>
    <li>
        <strong>Statistics on Customer:</strong> For each <code>customer ID</code>, we will compute:
        <ul>
            <li>The sum of all transaction amounts.</li>
            <li>The average amount per transaction.</li>
            <li>The total number of transactions.</li>
            <li>The number of days since the last transaction.</li>
        </ul>
    </li>
    <li>
        <strong>Spending Category Distribution:</strong> For each transaction category, we will compute:
        <ul>
            <li>The sum of transaction amounts.</li>
            <li>The mean, standard deviation, maximum, and median of transaction amounts.</li>
        </ul>
    </li>
</ol>

<p>These computations will result in two Teradata DataFrames:</p>
<ul>
    <li><code>df_eng_feat_cust</code>: Features computed per customer.</li>
    <li><code>df_eng_feat_cat</code>: Features computed for spending category distribution.</li>
</ul>

<p>Note that these DataFrames only implement the processing logic and do not generate data until explicitly stored or exported. When displaying the content of these DataFrames, only a sample of the results will be shown. To generate the actual data, you would need to either:</p>
<ul>
    <li>Store the results in another table within the database.</li>
    <li>Export the results to a <code>pandas</code> DataFrame, files, ...</li>
</ul>

## 5. Get Data For demo

### 5.1. Load the transaction Data

In [10]:
t_types = OrderedDict(CustomerID=INTEGER, Transaction_Amount=FLOAT, Date_transaction=DATE, Category=VARCHAR(200), MerchantID=INTEGER)
df = ctx.read_csv(table_name='transactions',
               filepath=r"data/transactions.csv",
              types=t_types)

### 5.2. Perform Data Transformation

In [11]:
df_eng_feat_cust = df.groupby('CustomerID').agg({'Transaction_Amount' : ['sum','mean','count'], 'Date_transaction':['max']})

#### Statistics on customers

In [12]:
df_eng_feat_cust = df.groupby('CustomerID').assign(total_Transaction_Amount=df.Transaction_Amount.sum(),
                                                   avg_Transaction_Amount=df.Transaction_Amount.mean(),
                                                   count_Transaction_Amount=df.Transaction_Amount.count(),
                                                   max_Date_transaction=df.Date_transaction.max()
                                                  )
df_eng_feat_cust = df_eng_feat_cust.assign(nb_days_since_last_transactions = literal_column('INTERVAL(PERIOD(max_Date_transaction, CURRENT_DATE)) DAY(4)',type_= INTEGER))
df_eng_feat_cust = df_eng_feat_cust[['CustomerID','total_Transaction_Amount','avg_Transaction_Amount','count_Transaction_Amount','nb_days_since_last_transactions']]
df_eng_feat_cust

CustomerID,total_Transaction_Amount,avg_Transaction_Amount,count_Transaction_Amount,nb_days_since_last_transactions
639481,76.89,76.89,1,1112
642109,198.73,198.73,1,1158
475918,74.36,74.36,1,1146
329746,295.67,295.67,1,1093
574328,145.52,145.52,1,1343
495867,56.49,56.49,1,1235
486735,72.83,72.83,1,1153
316427,312.54,312.54,1,1069
293658,345.76,345.76,1,1065
513867,92.15,92.15,1,1203


#### Spending Category Distribution

In [13]:
df_eng_feat_cat = df.groupby('Category').agg({'Transaction_Amount':['sum','mean','std','min','max','median']})
df_eng_feat_cat = df_eng_feat_cat.join(
    df[['Category','Transaction_Amount']].groupby("Category").percentile(0.25),
    on = 'Category',
    how = 'inner',
    rprefix = 'r'
)[df_eng_feat_cat.columns + ['percentile_Transaction_Amount']]
df_eng_feat_cat = df_eng_feat_cat.assign(quartile_1_Transaction_Amount=df_eng_feat_cat.percentile_Transaction_Amount)
df_eng_feat_cat = df_eng_feat_cat[[c for c in df_eng_feat_cat.columns if c not in ['percentile_Transaction_Amount']]]
df_eng_feat_cat = df_eng_feat_cat.join(
    df[['Category','Transaction_Amount']].groupby("Category").percentile(0.25),
    on = 'Category',
    how = 'inner',
    rprefix = 'r'
)[df_eng_feat_cat.columns + ['percentile_Transaction_Amount']]
df_eng_feat_cat = df_eng_feat_cat.assign(quartile_3_Transaction_Amount=df_eng_feat_cat.percentile_Transaction_Amount)
df_eng_feat_cat = df_eng_feat_cat[[c for c in df_eng_feat_cat.columns if c not in ['percentile_Transaction_Amount']]]
df_eng_feat_cat = df_eng_feat_cat.assign(var_feature = 'test')
df_eng_feat_cat

Category,sum_Transaction_Amount,mean_Transaction_Amount,std_Transaction_Amount,min_Transaction_Amount,max_Transaction_Amount,median_Transaction_Amount,quartile_1_Transaction_Amount,quartile_3_Transaction_Amount,var_feature
Cosmetic,2653.3999999999996,156.08235294117645,25.1113401995524,91.58,198.45,154.89,148.52,148.52,test
Market,1130.49,62.805,20.270501676951522,38.76,132.15,58.605000000000004,54.287499999999994,54.287499999999994,test
Travel,3127.0400000000004,173.72444444444446,245.85946313774275,72.94,1134.47,100.005,94.76,94.76,test
Restaurant,1455.4299999999996,76.6015789473684,15.967152137072615,57.44,134.76,75.42,70.6,70.6,test
Electronics,5242.8099999999995,308.4005882352941,65.75844522479497,198.67,456.83,295.73,278.34,278.34,test
Clothing,2426.0799999999995,142.7105882352941,51.63056234811275,89.42,287.39,125.69,118.73,118.73,test


## 6. Store the data transformations

We are storing the transformation here. So, even if underlying data varies, the data transformation steps remain same.

In [16]:
df_eng_feat_cust = df_eng_feat_cust.create_view('FEAT_ENG_CUST')
df_eng_feat_cust

CustomerID,total_Transaction_Amount,avg_Transaction_Amount,count_Transaction_Amount,nb_days_since_last_transactions
639481,76.89,76.89,1,1112
574328,145.52,145.52,1,1343
495867,56.49,56.49,1,1235
486735,72.83,72.83,1,1153
642109,198.73,198.73,1,1158
475918,74.36,74.36,1,1146
329746,295.67,295.67,1,1093
293658,345.76,345.76,1,1065
316427,312.54,312.54,1,1069
513867,92.15,92.15,1,1203


In [ ]:
df_eng_feat_cat = df_eng_feat_cat.create_view('FEAT_ENG_CAT')
df_eng_feat_cat

Category,sum_Transaction_Amount,mean_Transaction_Amount,std_Transaction_Amount,min_Transaction_Amount,max_Transaction_Amount,median_Transaction_Amount,quartile_1_Transaction_Amount,quartile_3_Transaction_Amount,var_feature
Market,1130.49,62.805,20.270501676951522,38.76,132.15,58.605000000000004,54.287499999999994,54.287499999999994,test
Cosmetic,2653.3999999999996,156.08235294117645,25.1113401995524,91.58,198.45,154.89,148.52,148.52,test
Restaurant,1455.4299999999996,76.6015789473684,15.967152137072615,57.44,134.76,75.42,70.6,70.6,test
Travel,3127.0400000000004,173.72444444444446,245.85946313774275,72.94,1134.47,100.005,94.76,94.76,test
Electronics,5242.8099999999995,308.4005882352941,65.75844522479497,198.67,456.83,295.73,278.34,278.34,test
Clothing,2426.0799999999995,142.7105882352941,51.63056234811275,89.42,287.39,125.69,118.73,118.73,test


## 7. Ingest the features from `df_eng_feat_cust` datasource

### 7.1. See the mind_map for Feature Store

Before ingesting the features, first check the mind map of feature store.
Since no features are ingested and no datasets are built, it should be empty.

In [ ]:
fs.mind_map()

### 7.2. Ingest the features

In [ ]:
# Ingest the features for data source `df_eng_feat_cust`.
fp = fs.get_feature_process(object=df_eng_feat_cust,
                            entity='CustomerID',
                            features=[
                                'total_Transaction_Amount', 
                                'avg_Transaction_Amount', 
                                'count_Transaction_Amount', 
                                'nb_days_since_last_transactions'],
                            description='Feature Process for Customers'
                           )
# Run it to ingest the features.
fp.run()

Process '77b35f75-c731-11f1-86f8-24eb16d29f8c' started.
Process '77b35f75-c731-11f1-86f8-24eb16d29f8c' completed.


True

### 7.3. See the mind_map for Feature Store

We ingested three features—`total_Transaction_Amount`, `avg_Transaction_Amount`, `count_Transaction_Amount` and `nb_days_since_last_transactions`—from a single feature process. This demonstrates how multiple related features can be managed and tracked together within the feature store, maintaining their lineage to the originating process.

In [ ]:
# Features are ingested. Before we move to other APIs, look at mind map.
fs.mind_map()

### 7.4. Explore FeatureStore for CustomerID entity

#### 7.4.1. list feature_processes

In [ ]:
# List the feature processes.
fs.list_feature_processes()

process_id,description,data_domain,process_type,data_source,entity_id,feature_names,feature_ids,valid_start,valid_end
77b35f75-c731-11f1-86f8-24eb16d29f8c,Feature Process for Customers,Analytics,denormalized view,"""ALICE"".""FEAT_ENG_CUST""",CustomerID,"avg_Transaction_Amount, count_Transaction_Amount, nb_days_since_last_transactions, total_Transaction_Amount",None,2026-10-13 18:11:28.130000+00:,9999-12-31 23:59:59.999999+00:


#### 7.4.2. list feature_catalogs

In [ ]:
# Look at feature catalogs.
fs.list_feature_catalogs()

entity_name,data_domain,feature_id,table_name,valid_start,valid_end
CustomerID,Analytics,2,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
CustomerID,Analytics,1,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
CustomerID,Analytics,4,FS_T_3937d0af_b408_517a_e99a_b6463b1e8e38,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
CustomerID,Analytics,3,FS_T_22063ce7_d537_561d_5cc0_b06120f5c2f0,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:


#### 7.4.3. list feature versions

##### 7.4.3.1. Get the feature catalog

In [ ]:
fc = fs.get_feature_catalog()

##### 7.4.3.2. list feature versions

In [ ]:
# Get the feature catalog and examine the versions.
fc.list_feature_versions()

entity_id,data_domain,id,name,table_name,feature_version
CustomerID,Analytics,2,avg_Transaction_Amount,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,77b35f75-c731-11f1-86f8-24eb16d29f8c
CustomerID,Analytics,4,nb_days_since_last_transactions,FS_T_3937d0af_b408_517a_e99a_b6463b1e8e38,77b35f75-c731-11f1-86f8-24eb16d29f8c
CustomerID,Analytics,3,count_Transaction_Amount,FS_T_22063ce7_d537_561d_5cc0_b06120f5c2f0,77b35f75-c731-11f1-86f8-24eb16d29f8c
CustomerID,Analytics,1,total_Transaction_Amount,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,77b35f75-c731-11f1-86f8-24eb16d29f8c


## 8. Ingest the features from `df_eng_feat_cat` datasource

### 8.1. Ingest the features

In [25]:
# Let's ingest features from another data source `df_eng_feat_cat`.
fp2 = fs.get_feature_process(object=df_eng_feat_cat,
                            entity='Category',
                            features=['sum_Transaction_Amount',
                                      'mean_Transaction_Amount',
                                      'std_Transaction_Amount',
                                      'min_Transaction_Amount',
                                      'max_Transaction_Amount',
                                      'median_Transaction_Amount',
                                      'quartile_1_Transaction_Amount',
                                      'quartile_3_Transaction_Amount',
                                      'var_feature'],
                            description='Feature Process for Category'
                           )
fp2.run()

Process '8dcd70b8-c731-11f1-a4e1-24eb16d29f8c' started.
Process '8dcd70b8-c731-11f1-a4e1-24eb16d29f8c' completed.


True

### 8.2. See the mind_map for Feature Store

In [26]:
# Let's look at mind map now.
fs.mind_map()

### 8.3. Explore the FeatureStore for the operations that have been done so far

#### 8.3.1. List feture processes

In [27]:
# List the feature process'es.
fs.list_feature_processes()

process_id,description,data_domain,process_type,data_source,entity_id,feature_names,feature_ids,valid_start,valid_end
77b35f75-c731-11f1-86f8-24eb16d29f8c,Feature Process for Customers,Analytics,denormalized view,"""ALICE"".""FEAT_ENG_CUST""",CustomerID,"avg_Transaction_Amount, count_Transaction_Amount, nb_days_since_last_transactions, total_Transaction_Amount",None,2026-10-13 18:11:28.130000+00:,9999-12-31 23:59:59.999999+00:
8dcd70b8-c731-11f1-a4e1-24eb16d29f8c,Feature Process for Category,Analytics,denormalized view,"""ALICE"".""FEAT_ENG_CAT""",Category,"max_Transaction_Amount, mean_Transaction_Amount, median_Transaction_Amount, min_Transaction_Amount, quartile_1_Transaction_Amount, quartile_3_Transaction_Amount, std_Transaction_Amount, sum_Transaction_Amount, var_feature",None,2026-10-13 18:12:19.020000+00:,9999-12-31 23:59:59.999999+00:


#### 8.3.2. List feature catalogs

In [28]:
# Look at feature catalogs.
display.max_rows=20
fs.list_feature_catalogs()

entity_name,data_domain,feature_id,table_name,valid_start,valid_end
Category,Analytics,10,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
Category,Analytics,13,FS_T_44b9b0fd_ee25_32c3_6ae6_c0df59351fd7,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
Category,Analytics,12,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
Category,Analytics,8,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
Category,Analytics,5,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
CustomerID,Analytics,3,FS_T_22063ce7_d537_561d_5cc0_b06120f5c2f0,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
CustomerID,Analytics,4,FS_T_3937d0af_b408_517a_e99a_b6463b1e8e38,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
CustomerID,Analytics,2,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
CustomerID,Analytics,1,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
Category,Analytics,6,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:


#### 8.3.3. List feature versions

In [29]:
# Get the feature catalog and examine the versions.
fc.list_feature_versions()

entity_id,data_domain,id,name,table_name,feature_version
CustomerID,Analytics,2,avg_Transaction_Amount,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,77b35f75-c731-11f1-86f8-24eb16d29f8c
Category,Analytics,6,mean_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,9,max_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,12,quartile_3_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,10,median_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,7,std_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,5,sum_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,11,quartile_1_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,13,var_feature,FS_T_44b9b0fd_ee25_32c3_6ae6_c0df59351fd7,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,8,min_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c


#### 8.3.4. List dataset catalogs

In [30]:
# Let's look at dataset catalogs. Since no dataset is created, it should be empty.
fs.list_dataset_catalogs()

id,data_domain,name,entity_name,database_name,description,valid_start,valid_end


## 9. Build Dataset

### 9.1. Get the dataset catalog

In [31]:
# Let's build dataset in Dataset catalog. First get the pointer for Dataset catalog.
dc = fs.get_dataset_catalog()

### 9.2. Build the dataset for 'df_eng_feat_cust'

In [32]:
df = dc.build_dataset(entity='CustomerID',
                      selected_features={'avg_Transaction_Amount': fp.process_id,
                                         'total_Transaction_Amount': fp.process_id},
                      view_name='CustID_Transactions_1'
                     )
df

CustomerID,avg_Transaction_Amount,total_Transaction_Amount
228043,118.87,118.87
464671,132.15,132.15
509676,117.67,117.67
329656,57.83,57.83
486735,72.83,72.83
329574,295.73,295.73
357264,173.94,173.94
495867,56.49,56.49
584729,59.73,59.73
396284,154.89,154.89


##### Verify the data. 

In [33]:
df_eng_feat_cust[df.CustomerID.isin([467220, 684935, 642109])]

CustomerID,total_Transaction_Amount,avg_Transaction_Amount,count_Transaction_Amount,nb_days_since_last_transactions
684935,125.89,125.89,1,1277
642109,198.73,198.73,1,1158
467220,129.35,129.35,1,1237


### 9.3. See the mind_map for Feature Store

In [34]:
# Let's look at mind map.
fs.mind_map()

### 9.4. Build the time series dataset for 'df_eng_feat_cust'

In [35]:
# Let's build a time series. 
df = dc.build_time_series(entity='Category',
                          selected_features={'std_Transaction_Amount': fp2.process_id,
                                             'max_Transaction_Amount': fp2.process_id,
                                             'median_Transaction_Amount': fp2.process_id,
                                             'quartile_3_Transaction_Amount': fp2.process_id},
                          view_name='Cat_Transactions_1'
                         )
df

Category,std_Transaction_Amount,std_Transaction_Amount_start_time,std_Transaction_Amount_end_time,max_Transaction_Amount,max_Transaction_Amount_start_time,max_Transaction_Amount_end_time,median_Transaction_Amount,median_Transaction_Amount_start_time,median_Transaction_Amount_end_time,quartile_3_Transaction_Amount,quartile_3_Transaction_Amount_start_time,quartile_3_Transaction_Amount_end_time
Cosmetic,25.1113401995524,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,198.45,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,154.89,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,148.52,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:
Travel,245.85946313774275,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,1134.47,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,100.005,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,94.76,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:
Clothing,51.63056234811275,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,287.39,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,125.69,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,118.73,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:
Restaurant,15.967152137072615,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,134.76,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,75.42,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,70.6,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:
Electronics,65.75844522479497,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,456.83,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,295.73,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,278.34,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:
Market,20.270501676951522,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,132.15,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,58.605000000000004,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:,54.287499999999994,2026-10-13 18:12:13.400000+00:,9999-12-31 23:59:59.999999+00:


### 9.5. See the mind_map for Feature Store

In [36]:
# Let's look at mind map.
fs.mind_map()

### 9.6. List dataset catalogs 

In [37]:
# Let's look at Dataset catalog. Two datasets created. Hence both should be available.
fs.list_dataset_catalogs()

id,data_domain,name,entity_name,database_name,description,valid_start,valid_end
6c6a452c-75d1-49a8-8f86-02239f4a887d,Analytics,CustID_Transactions_1,CustomerID,enterprise_feature_repo,None,2026-10-13 18:13:17.590000+00:,9999-12-31 23:59:59.999999+00:
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,Cat_Transactions_1,Category,enterprise_feature_repo,None,2026-10-13 18:13:38.230000+00:,9999-12-31 23:59:59.999999+00:


## 10. Feature Ingestion with filters

### 10.1. Load the sales data 

In [38]:
# Ingest the features with filters.
ctx.load_example_data('dataframe', 'sales')
df = ctx.DataFrame("sales")
df

accounts,Feb,Jan,Mar,Apr,datetime
Alpha Co,840.0,800,860,1000,04/01/2017
Red Inc,800.0,600,560,None,04/01/2017
Orange Inc,840.0,None,None,1000,04/01/2017
Jones LLC,800.0,600,560,720,04/01/2017
Yellow Inc,360.0,None,None,None,04/01/2017
Blue Inc,360.0,200,380,404,04/01/2017


### 10.2. Ingest the features

In [39]:
# Ingest the features only when entities has 'Inc' in it.
fp3 = fs.get_feature_process(object=df, 
                             entity='accounts', 
                             features=['Jan', 'Feb'])
fp3.run(filters=df.accounts.str.contains('Inc')==1)

Process 'dba30426-c731-11f1-a4d3-24eb16d29f8c' started.
Ingesting the features for filter 'CASE WHEN (accounts IS NULL) THEN NULL ELSE CASE WHEN (regexp_substr(accounts, 'Inc', 1, 1, 'c') IS NULL) THEN 0 ELSE 1 END END = 1' to catalog.
Process 'dba30426-c731-11f1-a4d3-24eb16d29f8c' completed.


True

### 10.3. See the mind_map for Feature Store

In [40]:
# Let's look at mind map to see features are ingested or not. 
fs.mind_map()

### 10.4. Look at the feature table 

In [41]:
# Let's verify whether features ingested for that features or not. Look at catalog table.
fc.list_feature_versions()

entity_id,data_domain,id,name,table_name,feature_version
CustomerID,Analytics,2,avg_Transaction_Amount,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,77b35f75-c731-11f1-86f8-24eb16d29f8c
accounts,Analytics,14,Jan,FS_T_4fb46368_0c60_21bf_20a1_b0a3948fedde,dba30426-c731-11f1-a4d3-24eb16d29f8c
CustomerID,Analytics,4,nb_days_since_last_transactions,FS_T_3937d0af_b408_517a_e99a_b6463b1e8e38,77b35f75-c731-11f1-86f8-24eb16d29f8c
Category,Analytics,6,mean_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,12,quartile_3_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,8,min_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,10,median_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,7,std_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,5,sum_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,11,quartile_1_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c


In [42]:
ctx.DataFrame(ctx.in_schema(fs.repo, "FS_T_4fb46368_0c60_21bf_20a1_b0a3948fedde"))

accounts,feature_id,feature_value,feature_version,valid_start,valid_end,ValidPeriod
Red Inc,14,600,dba30426-c731-11f1-a4d3-24eb16d29f8c,2026-10-13 18:14:10.630000+00:,9999-12-31 23:59:59.999999+00:,('2026-10-13 18:14:10.630000+0
Orange Inc,14,None,dba30426-c731-11f1-a4d3-24eb16d29f8c,2026-10-13 18:14:10.630000+00:,9999-12-31 23:59:59.999999+00:,('2026-10-13 18:14:10.630000+0
Yellow Inc,14,None,dba30426-c731-11f1-a4d3-24eb16d29f8c,2026-10-13 18:14:10.630000+00:,9999-12-31 23:59:59.999999+00:,('2026-10-13 18:14:10.630000+0
Blue Inc,14,200,dba30426-c731-11f1-a4d3-24eb16d29f8c,2026-10-13 18:14:10.630000+00:,9999-12-31 23:59:59.999999+00:,('2026-10-13 18:14:10.630000+0


### 10.5. Ingest the features for an incremental load.

#### 10.5.1. let's first update the values of features. Otherwise, update is not visible.

In [43]:
ctx.execute_sql('update sales set Feb=Feb*2')

TeradataCursor uRowsHandle=618 bClosed=False

#### 10.5.2. Store the values for increamental load

In [44]:
fp4 = fs.get_feature_process(object=df, 
                             entity='accounts', 
                             features=['Jan', 'Feb']
                            )
fp4.run()

Process 'dba30426-c731-11f1-a4d3-24eb16d29f8c' started.
Process 'dba30426-c731-11f1-a4d3-24eb16d29f8c' completed.


True

### 10.6. Build the dataset

In [45]:
# Let's first look at the data. For that, let's build a dataset.
df_sales_jan_feb = dc.build_dataset(entity='accounts',
                                    selected_features={
                                        'Jan': fp4.process_id,
                                        'Feb': fp4.process_id
                                    },
                                   view_name='sales_jan_feb')

In [46]:
df_sales_jan_feb.sort('accounts')

accounts,Jan,Feb
Alpha Co,800,1680.0
Blue Inc,200,720.0
Jones LLC,600,1600.0
Orange Inc,None,1680.0
Red Inc,600,1600.0
Yellow Inc,None,720.0


## 11. Ingest the features at specific time

### 11.1. Create new data domain for specific time ingestion

In [47]:
# For that, let's create another data domain.
fs_sales_analytics = ctx.FeatureStore(repo='enterprise_feature_repo', data_domain='Sales_Analytics')

FeatureStore is ready to use.


### 11.2. See the FeatureStore for 'Sales_Analytics' data domain

In [48]:
# Let's first see what this data domain has. It should be empty.
fs_sales_analytics.mind_map()

### 11.3. Drop and load the sales data

In [49]:
# Let's drop the sales table and load it again.
ctx.db_drop_table('sales')
ctx.load_example_data('dataframe', 'sales')
df = ctx.DataFrame("sales")

### 11.4. Ingest the features

In [50]:
# Let's first ingest the features.
fp5 = fs_sales_analytics.get_feature_process(object=df, 
                                             entity='accounts', 
                                             features=['Jan', 'Feb', 'Mar', 'Apr']
                                             )
fp5.run()

Process '0a349e81-c732-11f1-80a4-24eb16d29f8c' started.
Process '0a349e81-c732-11f1-80a4-24eb16d29f8c' completed.


True

In [51]:
fs_sales_analytics.mind_map()

### 11.5. Look at Jan's values first.

In [52]:
fc_sales_analytics = fs_sales_analytics.get_feature_catalog()
fc_sales_analytics.list_feature_versions()

entity_id,data_domain,id,name,table_name,feature_version
accounts,Sales_Analytics,16,Jan,FS_T_feb531d5_375e_5831_05d7_3ac2ac2a7965,0a349e81-c732-11f1-80a4-24eb16d29f8c
accounts,Sales_Analytics,19,Apr,FS_T_feb531d5_375e_5831_05d7_3ac2ac2a7965,0a349e81-c732-11f1-80a4-24eb16d29f8c
accounts,Sales_Analytics,17,Feb,FS_T_9a644a9e_8cf7_5ac4_8c06_a621b7111619,0a349e81-c732-11f1-80a4-24eb16d29f8c
accounts,Sales_Analytics,18,Mar,FS_T_feb531d5_375e_5831_05d7_3ac2ac2a7965,0a349e81-c732-11f1-80a4-24eb16d29f8c


In [53]:
# Let's look at Jan's values first.
sales_jan = ctx.DataFrame(ctx.in_schema(fs.repo, "FS_T_feb531d5_375e_5831_05d7_3ac2ac2a7965")).as_of(valid_time='current')
sales_jan[sales_jan.feature_id == 16]

accounts,feature_id,feature_value,feature_version,valid_start,valid_end
Blue Inc,16,50,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:
Alpha Co,16,200,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:
Jones LLC,16,150,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:
Yellow Inc,16,None,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:
Orange Inc,16,None,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:
Red Inc,16,150,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:


### 11.5. Update the feature values to simulate the data change. 

In [54]:
ctx.execute_sql('update sales set Jan=Jan*2, Feb=Feb*2, Mar=Mar*2, Apr=Apr*2')

TeradataCursor uRowsHandle=837 bClosed=False

### 11.6. Ingest the same features again

In [55]:
fp6 = fs_sales_analytics.get_feature_process(object=df, 
                                             entity='accounts', 
                                             features=['Jan', 'Feb', 'Mar', 'Apr']
                                             )
fp6.run()

Process '0a349e81-c732-11f1-80a4-24eb16d29f8c' started.
Process '0a349e81-c732-11f1-80a4-24eb16d29f8c' completed.


True

In [56]:
# Let's look at mind map. No new feature process initiated. Hence it should be same as previous state.
fs_sales_analytics.mind_map()

#### look at features again. Observe that only the new values appeared.

**Note:** Old values are not visible since function as_of selected only current records.

In [57]:
sales_jan = ctx.DataFrame(ctx.in_schema(fs.repo, "FS_T_feb531d5_375e_5831_05d7_3ac2ac2a7965")).as_of(valid_time='current')
sales_jan[sales_jan.feature_id == 16]

accounts,feature_id,feature_value,feature_version,valid_start,valid_end
Blue Inc,16,100,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Red Inc,16,300,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Orange Inc,16,None,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:
Yellow Inc,16,None,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:
Jones LLC,16,300,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Alpha Co,16,400,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:


#### Select all values. Then notice the difference.

In [58]:
sales_jan = ctx.DataFrame(ctx.in_schema(fs.repo, "FS_T_feb531d5_375e_5831_05d7_3ac2ac2a7965")).as_of(valid_time=None)
sales_jan[sales_jan.feature_id == 16].sort(['accounts', 'valid_start'])

accounts,feature_id,feature_value,feature_version,valid_start,valid_end
Alpha Co,16,200,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,2026-10-13 18:16:02.600000+00:
Alpha Co,16,400,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Blue Inc,16,50,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,2026-10-13 18:16:02.600000+00:
Blue Inc,16,100,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Jones LLC,16,150,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,2026-10-13 18:16:02.600000+00:
Jones LLC,16,300,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Orange Inc,16,None,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:
Red Inc,16,150,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,2026-10-13 18:16:02.600000+00:
Red Inc,16,300,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Yellow Inc,16,None,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:


### 11.7. Update the data again to ingest for a specific time.

In [59]:
# Double the data again. This time ingest for a specific time.
# Update the feature values to simulate the data change. 
ctx.execute_sql('update sales set Jan=Jan*2, Feb=Feb*2, Mar=Mar*2, Apr=Apr*2')

TeradataCursor uRowsHandle=922 bClosed=False

### 11.8. Ingest feature value for specific time

In [60]:
# Let's first ingest the features.
fp7 = fs_sales_analytics.get_feature_process(object=df, 
                                             entity='accounts', 
                                             features=['Jan', 'Feb', 'Mar', 'Apr'],
                                             )
fp7.run(as_of='2025-09-08 04:51:59.000000+00:00')

Process '0a349e81-c732-11f1-80a4-24eb16d29f8c' started.
Process '0a349e81-c732-11f1-80a4-24eb16d29f8c' completed.


True

In [61]:
# Let's verify the data again.
sales_jan = ctx.DataFrame(ctx.in_schema(fs.repo, "FS_T_feb531d5_375e_5831_05d7_3ac2ac2a7965")).as_of(valid_time=None)
sales_jan[sales_jan.feature_id == 16].sort(['accounts', 'valid_start'])

accounts,feature_id,feature_value,feature_version,valid_start,valid_end
Alpha Co,16,800,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,2026-10-13 18:16:02.600000+00:
Alpha Co,16,800,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Blue Inc,16,200,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,2026-10-13 18:16:02.600000+00:
Blue Inc,16,200,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Jones LLC,16,600,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,2026-10-13 18:16:02.600000+00:
Jones LLC,16,600,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Orange Inc,16,None,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:
Red Inc,16,600,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,2026-10-13 18:16:02.600000+00:
Red Inc,16,600,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:16:02.600000+00:,9999-12-31 23:59:59.999999+00:
Yellow Inc,16,None,0a349e81-c732-11f1-80a4-24eb16d29f8c,2026-10-13 18:15:33.600000+00:,9999-12-31 23:59:59.999999+00:


### 11.9. See the mind_map for fs_sales_analytics FeatureStore

In [62]:
# Remove the whole data domain.
# Before removing it, Let's first look at feature store for this data domain.
fs_sales_analytics.mind_map()

## 12. Remove 'Sales_Analytics' DataDomain

### 12.1. Remove DataDomain

In [63]:
# Remove it and then again look at mind map.
fs_sales_analytics.remove_data_domain()

Data domain 'Sales_Analytics' is removed from the FeatureStore.


True

### 12.2. See the FeatureStore for Sales_analytics data domain

In [64]:
# Verify the objects.
fs_sales_analytics.mind_map()

### 12.3. See the FeatureStore Analytics data domain

In [65]:
# Also verify the objects of fs. Since only data domain sales analytics is removed, fs objects should remain as it is.
fs.mind_map()

## 13. Dataset Management

### 13.1. List datasets 

In [66]:
dc.list_datasets()

id,data_domain,name,entity_name,description,valid_start,valid_end
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,Cat_Transactions_1,Category,None,2026-10-13 18:13:38.230000+00:,9999-12-31 23:59:59.999999+00:
cf3f99b7-b3f6-4388-a57e-b95ab2e898e0,Analytics,sales_jan_feb,accounts,None,2026-10-13 18:14:57.950000+00:,9999-12-31 23:59:59.999999+00:
6c6a452c-75d1-49a8-8f86-02239f4a887d,Analytics,CustID_Transactions_1,CustomerID,None,2026-10-13 18:13:17.590000+00:,9999-12-31 23:59:59.999999+00:


### 13.2. List features

In [67]:
dc.list_features()

dataset_id,data_domain,feature_id,feature_name,feature_view
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,9,max_Transaction_Amount,Cat_Transactions_1
6c6a452c-75d1-49a8-8f86-02239f4a887d,Analytics,1,total_Transaction_Amount,CustID_Transactions_1
6c6a452c-75d1-49a8-8f86-02239f4a887d,Analytics,2,avg_Transaction_Amount,CustID_Transactions_1
cf3f99b7-b3f6-4388-a57e-b95ab2e898e0,Analytics,15,Feb,sales_jan_feb
cf3f99b7-b3f6-4388-a57e-b95ab2e898e0,Analytics,14,Jan,sales_jan_feb
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,7,std_Transaction_Amount,Cat_Transactions_1
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,12,quartile_3_Transaction_Amount,Cat_Transactions_1
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,10,median_Transaction_Amount,Cat_Transactions_1


### 13.2. List Entities 

In [68]:
dc.list_entities()

id,data_domain,name,entity_name,description
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,Cat_Transactions_1,Category,None
cf3f99b7-b3f6-4388-a57e-b95ab2e898e0,Analytics,sales_jan_feb,accounts,None
6c6a452c-75d1-49a8-8f86-02239f4a887d,Analytics,CustID_Transactions_1,CustomerID,None


### 13.4.Acrhive datasets

In [70]:
dc.archive_datasets('cf3f99b7-b3f6-4388-a57e-b95ab2e898e0')

Dataset id(s) 'cf3f99b7-b3f6-4388-a57e-b95ab2e898e0' is/are archived from the dataset catalog.


True

In [71]:
dc.list_datasets()

id,data_domain,name,entity_name,description,valid_start,valid_end
cf3f99b7-b3f6-4388-a57e-b95ab2e898e0,Analytics,sales_jan_feb,accounts,None,2026-10-13 18:14:57.950000+00:,2026-10-13 18:27:38.680000+00:
6c6a452c-75d1-49a8-8f86-02239f4a887d,Analytics,CustID_Transactions_1,CustomerID,None,2026-10-13 18:13:17.590000+00:,9999-12-31 23:59:59.999999+00:
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,Cat_Transactions_1,Category,None,2026-10-13 18:13:38.230000+00:,9999-12-31 23:59:59.999999+00:


### 13.5. Delete datasets

In [72]:
# Delete the archived dataset.
dc.delete_datasets('cf3f99b7-b3f6-4388-a57e-b95ab2e898e0')

Dataset id(s) 'cf3f99b7-b3f6-4388-a57e-b95ab2e898e0' is/are deleted from the dataset catalog.


True

In [73]:
# Verify the datasets after deleting dataset 02eadbbb-aa37-431c-9489-92fc30f386b8.
dc.list_datasets()

id,data_domain,name,entity_name,description,valid_start,valid_end
6c6a452c-75d1-49a8-8f86-02239f4a887d,Analytics,CustID_Transactions_1,CustomerID,None,2026-10-13 18:13:17.590000+00:,9999-12-31 23:59:59.999999+00:
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,Cat_Transactions_1,Category,None,2026-10-13 18:13:38.230000+00:,9999-12-31 23:59:59.999999+00:


### 13.6. List features

In [74]:
# Verify the features after deleting dataset 02eadbbb-aa37-431c-9489-92fc30f386b8.
dc.list_features()

dataset_id,data_domain,feature_id,feature_name,feature_view
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,9,max_Transaction_Amount,Cat_Transactions_1
6c6a452c-75d1-49a8-8f86-02239f4a887d,Analytics,1,total_Transaction_Amount,CustID_Transactions_1
6c6a452c-75d1-49a8-8f86-02239f4a887d,Analytics,2,avg_Transaction_Amount,CustID_Transactions_1
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,7,std_Transaction_Amount,Cat_Transactions_1
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,12,quartile_3_Transaction_Amount,Cat_Transactions_1
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,10,median_Transaction_Amount,Cat_Transactions_1


### 13.7. List entities

In [75]:
# Verify the Entities after deleting dataset 02eadbbb-aa37-431c-9489-92fc30f386b8.
dc.list_entities()

id,data_domain,name,entity_name,description
6c6a452c-75d1-49a8-8f86-02239f4a887d,Analytics,CustID_Transactions_1,CustomerID,None
b9a5411f-6415-459c-8959-683222a6d9cc,Analytics,Cat_Transactions_1,Category,None


### 13.8. See mind_map for FeatureStore after deleting dataset

In [76]:
# Let's look at mind map of Feature store now. Deleted dataset should not appear.
fs.mind_map()

## 14. Dataset

### 14.1. Explore Dataset methods

#### 14.1.1. Get the dataset

In [77]:
ds = dc.get_dataset('6c6a452c-75d1-49a8-8f86-02239f4a887d')

### 14.2. Explore Properties

#### 14.2.1. features

In [78]:
ds.features

[Feature(name=total_Transaction_Amount), Feature(name=avg_Transaction_Amount)]

#### 14.2.2. entity

In [79]:
ds.entity

[Entity(name=CustomerID)]

#### 14.2.3. id

In [80]:
ds.id

'6c6a452c-75d1-49a8-8f86-02239f4a887d'

#### 14.2.4. view_name

In [81]:
ds.view_name

'CustID_Transactions_1'

## 15. Feature Management

### 15.1. List features

In [82]:
fc.list_features()

feature_id,name,entity_name,data_type,feature_type,valid_start,valid_end
9,max_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
3,count_Transaction_Amount,CustomerID,BIGINT,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
1,total_Transaction_Amount,CustomerID,FLOAT,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
10,median_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
2,avg_Transaction_Amount,CustomerID,FLOAT,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
15,Feb,accounts,FLOAT,CONTINUOUS,2026-10-13 18:14:06.340000+00:,9999-12-31 23:59:59.999999+00:
14,Jan,accounts,BIGINT,CONTINUOUS,2026-10-13 18:14:06.340000+00:,9999-12-31 23:59:59.999999+00:
6,mean_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
7,std_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
12,quartile_3_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:


### 15.2. list feature versions

In [83]:
fc.list_feature_versions()

entity_id,data_domain,id,name,table_name,feature_version
Category,Analytics,12,quartile_3_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,10,median_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,7,std_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,5,sum_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,13,var_feature,FS_T_44b9b0fd_ee25_32c3_6ae6_c0df59351fd7,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
CustomerID,Analytics,1,total_Transaction_Amount,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,77b35f75-c731-11f1-86f8-24eb16d29f8c
CustomerID,Analytics,3,count_Transaction_Amount,FS_T_22063ce7_d537_561d_5cc0_b06120f5c2f0,77b35f75-c731-11f1-86f8-24eb16d29f8c
CustomerID,Analytics,2,avg_Transaction_Amount,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,77b35f75-c731-11f1-86f8-24eb16d29f8c
accounts,Analytics,15,Feb,FS_T_62c9e14a_e0ca_d196_bca1_cc3436ebc70d,dba30426-c731-11f1-a4d3-24eb16d29f8c
accounts,Analytics,14,Jan,FS_T_4fb46368_0c60_21bf_20a1_b0a3948fedde,dba30426-c731-11f1-a4d3-24eb16d29f8c


In [84]:
# Looks to be a bug. Investigation is in progress.
fc.features

[Feature(name=median_Transaction_Amount),
 Feature(name=count_Transaction_Amount),
 Feature(name=quartile_1_Transaction_Amount),
 Feature(name=var_feature),
 Feature(name=nb_days_since_last_transactions),
 Feature(name=total_Transaction_Amount),
 Feature(name=Feb),
 Feature(name=avg_Transaction_Amount),
 Feature(name=mean_Transaction_Amount),
 Feature(name=Jan),
 Feature(name=max_Transaction_Amount),
 Feature(name=quartile_3_Transaction_Amount),
 Feature(name=std_Transaction_Amount),
 Feature(name=sum_Transaction_Amount),
 Feature(name=min_Transaction_Amount)]

In [85]:
fc.entities

[Entity(name=Category), Entity(name=CustomerID), Entity(name=accounts)]

### 15.3. Archive features

In [86]:
# Let's archive a feature which is associated with any dataset. 
fc.archive_features('Jan')

Feature 'Jan' is archived from table 'FS_T_4fb46368_0c60_21bf_20a1_b0a3948fedde'.
Feature 'Jan' is archived from metadata.


True

In [87]:
# Observe that the Jan record is closed.
fc.list_features()

feature_id,name,entity_name,data_type,feature_type,valid_start,valid_end
6,mean_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
12,quartile_3_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
4,nb_days_since_last_transactions,CustomerID,INTERVAL_DAY,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
13,var_feature,Category,VARCHAR,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
9,max_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
5,sum_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
3,count_Transaction_Amount,CustomerID,BIGINT,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
1,total_Transaction_Amount,CustomerID,FLOAT,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
10,median_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
8,min_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:


### 15.3. Look at the feature values also. Even they all should be closed.

In [88]:
# Look at the feature values also. Even they all should be closed.
display.max_rows=30
fc.list_feature_versions()

entity_id,data_domain,id,name,table_name,feature_version
Category,Analytics,12,quartile_3_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,10,median_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,7,std_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,5,sum_Transaction_Amount,FS_T_9e4a4f56_14ff_7629_68eb_ce1cfa28c901,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
Category,Analytics,13,var_feature,FS_T_44b9b0fd_ee25_32c3_6ae6_c0df59351fd7,8dcd70b8-c731-11f1-a4e1-24eb16d29f8c
CustomerID,Analytics,1,total_Transaction_Amount,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,77b35f75-c731-11f1-86f8-24eb16d29f8c
CustomerID,Analytics,3,count_Transaction_Amount,FS_T_22063ce7_d537_561d_5cc0_b06120f5c2f0,77b35f75-c731-11f1-86f8-24eb16d29f8c
CustomerID,Analytics,2,avg_Transaction_Amount,FS_T_14603143_aa22_6a4d_42f7_adfb07356c0c,77b35f75-c731-11f1-86f8-24eb16d29f8c
accounts,Analytics,15,Feb,FS_T_62c9e14a_e0ca_d196_bca1_cc3436ebc70d,dba30426-c731-11f1-a4d3-24eb16d29f8c
CustomerID,Analytics,4,nb_days_since_last_transactions,FS_T_3937d0af_b408_517a_e99a_b6463b1e8e38,77b35f75-c731-11f1-86f8-24eb16d29f8c


In [89]:
ndf = ctx.DataFrame(ctx.in_schema(fs.repo, "FS_T_4fb46368_0c60_21bf_20a1_b0a3948fedde")).as_of(valid_time='current')
ndf[ndf.feature_id==14]

accounts,feature_id,feature_value,feature_version,valid_start,valid_end


In [90]:
# Retrieve all the records. Since Jan is still not deleted, as_of with valid_time as None should still show results.
# However, all the records should be closed.
ndf = ctx.DataFrame(ctx.in_schema(fs.repo, "FS_T_4fb46368_0c60_21bf_20a1_b0a3948fedde")).as_of(valid_time=None)
ndf[ndf.feature_id==14]

accounts,feature_id,feature_value,feature_version,valid_start,valid_end
Alpha Co,14,800,dba30426-c731-11f1-a4d3-24eb16d29f8c,2026-10-13 18:14:40.450000+00:,2026-10-13 18:29:55.590000+00:
Blue Inc,14,200,dba30426-c731-11f1-a4d3-24eb16d29f8c,2026-10-13 18:14:10.630000+00:,2026-10-13 18:29:55.590000+00:
Yellow Inc,14,None,dba30426-c731-11f1-a4d3-24eb16d29f8c,2026-10-13 18:14:10.630000+00:,2026-10-13 18:29:55.590000+00:
Jones LLC,14,600,dba30426-c731-11f1-a4d3-24eb16d29f8c,2026-10-13 18:14:40.450000+00:,2026-10-13 18:29:55.590000+00:
Orange Inc,14,None,dba30426-c731-11f1-a4d3-24eb16d29f8c,2026-10-13 18:14:10.630000+00:,2026-10-13 18:29:55.590000+00:
Red Inc,14,600,dba30426-c731-11f1-a4d3-24eb16d29f8c,2026-10-13 18:14:10.630000+00:,2026-10-13 18:29:55.590000+00:


### 15.4. Deletes features

In [91]:
# Let's delete the archived feature.
fc.delete_features('Jan')

Feature 'Jan' is deleted from table 'FS_T_4fb46368_0c60_21bf_20a1_b0a3948fedde'.
Feature 'Jan' is deleted from metadata.
Table 'FS_T_4fb46368_0c60_21bf_20a1_b0a3948fedde' is dropped as it is not referenced in metadata.


True

In [92]:
# Verify Jan is completly removed or not.
fc.list_features()

feature_id,name,entity_name,data_type,feature_type,valid_start,valid_end
9,max_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
3,count_Transaction_Amount,CustomerID,BIGINT,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
1,total_Transaction_Amount,CustomerID,FLOAT,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
10,median_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
2,avg_Transaction_Amount,CustomerID,FLOAT,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
7,std_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
12,quartile_3_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
4,nb_days_since_last_transactions,CustomerID,INTERVAL_DAY,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
15,Feb,accounts,FLOAT,CONTINUOUS,2026-10-13 18:14:06.340000+00:,9999-12-31 23:59:59.999999+00:
6,mean_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:


### 15.5. Let's look at the mind_map now

In [93]:
# Let's look at mind_map now. 
fs.mind_map()

### 15.6. List feature processes

In [94]:
# Let's remove the feature process. First look at feature processes.
fs.list_feature_processes()

process_id,description,data_domain,process_type,data_source,entity_id,feature_names,feature_ids,valid_start,valid_end
77b35f75-c731-11f1-86f8-24eb16d29f8c,Feature Process for Customers,Analytics,denormalized view,"""ALICE"".""FEAT_ENG_CUST""",CustomerID,"avg_Transaction_Amount, count_Transaction_Amount, nb_days_since_last_transactions, total_Transaction_Amount",None,2026-10-13 18:11:28.130000+00:,9999-12-31 23:59:59.999999+00:
8dcd70b8-c731-11f1-a4e1-24eb16d29f8c,Feature Process for Category,Analytics,denormalized view,"""ALICE"".""FEAT_ENG_CAT""",Category,"max_Transaction_Amount, mean_Transaction_Amount, median_Transaction_Amount, min_Transaction_Amount, quartile_1_Transaction_Amount, quartile_3_Transaction_Amount, std_Transaction_Amount, sum_Transaction_Amount, var_feature",None,2026-10-13 18:12:19.020000+00:,9999-12-31 23:59:59.999999+00:
dba30426-c731-11f1-a4d3-24eb16d29f8c,,Analytics,denormalized view,"""sales""",accounts,"Feb, Jan",None,2026-10-13 18:14:12.500000+00:,9999-12-31 23:59:59.999999+00:


### 15.7. Achive feature process

In [95]:
fs.archive_feature_process('dba30426-c731-11f1-a4d3-24eb16d29f8c')

Feature 'Feb' is archived from table 'FS_T_62c9e14a_e0ca_d196_bca1_cc3436ebc70d'.
Feature 'Feb' is archived from metadata.
Feature 'Jan' does not exist in feature catalog.
FeatureProcess with process id 'dba30426-c731-11f1-a4d3-24eb16d29f8c' is archived.


False

In [96]:
# Let's look at feature processes. Observe that feature process '85dc42e8-670f-11f0-bb98-c934b24a960f' is also archived. It is closed.
fs.list_feature_processes()

process_id,description,data_domain,process_type,data_source,entity_id,feature_names,feature_ids,valid_start,valid_end
77b35f75-c731-11f1-86f8-24eb16d29f8c,Feature Process for Customers,Analytics,denormalized view,"""ALICE"".""FEAT_ENG_CUST""",CustomerID,"avg_Transaction_Amount, count_Transaction_Amount, nb_days_since_last_transactions, total_Transaction_Amount",None,2026-10-13 18:11:28.130000+00:,9999-12-31 23:59:59.999999+00:
8dcd70b8-c731-11f1-a4e1-24eb16d29f8c,Feature Process for Category,Analytics,denormalized view,"""ALICE"".""FEAT_ENG_CAT""",Category,"max_Transaction_Amount, mean_Transaction_Amount, median_Transaction_Amount, min_Transaction_Amount, quartile_1_Transaction_Amount, quartile_3_Transaction_Amount, std_Transaction_Amount, sum_Transaction_Amount, var_feature",None,2026-10-13 18:12:19.020000+00:,9999-12-31 23:59:59.999999+00:
dba30426-c731-11f1-a4d3-24eb16d29f8c,,Analytics,denormalized view,"""sales""",accounts,"Feb, Jan",None,2026-10-13 18:14:12.500000+00:,2026-10-13 18:34:44.150000+00:


In [97]:
# Let's look at catalog data. Observe that feature Feb is also archived. It is closed.
fc.list_features()

feature_id,name,entity_name,data_type,feature_type,valid_start,valid_end
4,nb_days_since_last_transactions,CustomerID,INTERVAL_DAY,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
11,quartile_1_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
9,max_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
5,sum_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
1,total_Transaction_Amount,CustomerID,FLOAT,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
10,median_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
8,min_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:
2,avg_Transaction_Amount,CustomerID,FLOAT,CONTINUOUS,2026-10-13 18:11:22.230000+00:,9999-12-31 23:59:59.999999+00:
15,Feb,accounts,FLOAT,CONTINUOUS,2026-10-13 18:14:06.340000+00:,2026-10-13 18:34:44.150000+00:
6,mean_Transaction_Amount,Category,FLOAT,CONTINUOUS,2026-10-13 18:12:09.360000+00:,9999-12-31 23:59:59.999999+00:


### 15.8. Delete feature process

In [98]:
# Delete the archived feature process.
fs.delete_feature_process('dba30426-c731-11f1-a4d3-24eb16d29f8c')

Feature 'Feb' is deleted from table 'FS_T_62c9e14a_e0ca_d196_bca1_cc3436ebc70d'.
Feature 'Feb' is deleted from metadata.
Feature 'Jan' does not exist in feature catalog.
Table 'FS_T_62c9e14a_e0ca_d196_bca1_cc3436ebc70d' is dropped as it is not referenced in metadata.
FeatureProcess with process id 'dba30426-c731-11f1-a4d3-24eb16d29f8c' is deleted.


False

### 15.9. Let's look at mind map now.

In [99]:
fs.mind_map()

In [100]:
# look at mind map now. All should be empty.
fs_sales_analytics.mind_map()

### 15.10. Remove data domain

In [104]:
# Let's remove the whole data domain. That should remove all objects.
fs.remove_data_domain()

Data domain 'Analytics' is removed from the FeatureStore.


True

## 16. Cleanup 

### 16.1. Drop views

In [102]:
ctx.db_drop_view('FEAT_ENG_CUST')

True

In [103]:
ctx.db_drop_view('FEAT_ENG_CAT')

True

### 16.2. Drop table

In [105]:
ctx.db_drop_table('transactions')

True

In [106]:
ctx.db_drop_table('sales')

True

### 16.3. Remove the Context

In [107]:
# Let's remove the feature store. First, disconnect from user and connect with DB Admin. 
# Removing Feature Store drops the database and all associated objects. So, doing it only with DB Admin.
# First disconnect from current user and establish connection with DBAdmin.
ctx.remove_context()

True

### 16.3. Delete Feature Store

In [108]:
ctx = create_context(config_file='admin_config_file.env')

**Note** : This will drop the database if all objects are removed.

In [109]:
fs = ctx.FeatureStore(repo='enterprise_feature_repo', data_domain='Analytics')
fs.delete()

FeatureStore is ready to use.


True

In [110]:
# Let's verify the repos. It should be empty.
ctx.FeatureStore.list_repos()

repos
test_fs
vfs_test_v1_obj
vfs_test
テストリポジトリ


In [111]:
# Finally disconnect the DB Admin from Vantage.
ctx.remove_context()

True